# Stage 3 — DINOv2 fingertip-pool + temporal context + visibility gate (5-fold subject CV)

Tests the strengthened fingertip-pool design after Stage 2 (mean-pool) failed at CER 0.86 and Stage 1 v3 falsified the DANN hypothesis.

**Recipe** (locked from Stage 1 v2; see `configs/stage3.yaml` for the contract):
- DINOv2-S/14 at **336x336** → 24x24 = 576 patches per frame.
- **Bell-weighted 3x3 patch window** around MediaPipe `INDEX_FINGER_TIP` (joint 8), edge-clamped.
- **±1 temporal context concat** → 3·D = 1152 channels.
- **Visibility gate**: zero the context block on MediaPipe dropouts and append a 0/1 vis bit → final per-frame feature is **1153-dim**.
- Pre-projection `LayerNorm(1153)` in the Conformer.  Optimiser / scheduler / augmentation / seed / Conformer config = Stage 1 v2.

**Pass/fail** (per plan §2 Task B):
- Every fold must reach **train NLL < 0.5** within 80 epochs — gating test.
- Headline: mean val CER ≤ **0.860** (beat Stage 2).
- Stretch: mean val CER ≤ **0.78** (Stage 4 fusion is likely to clear 0.55).

**Expected wall-clock on a single GPU**:

| GPU | Cache build | 5-fold sweep |
|---|---|---|
| T4  | ~50 min | ~12 h |
| L4  | ~25 min | ~5 h  |
| A100 | ~12 min | ~2.5 h |


## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy transformers --quiet

import sys, os
BASE = '/content'
WITA_ROOT = os.path.join(BASE, 'wita_v2')
!rm -rf {WITA_ROOT}
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" {WITA_ROOT}
sys.path.insert(0, BASE)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU            : {name}  ({mem:.1f} GB)')

## Cell 2 — Optional Drive mount + config (Stage 3 hyperparams)

`USE_DRIVE = True` mounts Google Drive so the DINOv2 fingertip cache (slow to build) and the per-fold results survive runtime disconnects.

In [ ]:
import os, logging, random, json
import numpy as np
import torch

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PERSIST_ROOT = '/content/drive/MyDrive/wita_v2_stage3'
else:
    PERSIST_ROOT = '/content/wita_v2_stage3'
os.makedirs(PERSIST_ROOT, exist_ok=True)

STAGE3_CACHE = os.path.join(PERSIST_ROOT, 'stage3_features.pt')
CV_MANIFEST  = os.path.join(PERSIST_ROOT, 'subject_cv5.json')
RESULTS_PATH = os.path.join(PERSIST_ROOT, 'stage3_results.json')

EPHEM = '/content'
os.makedirs(os.path.join(EPHEM, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(EPHEM, 'logs'),        exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(os.path.join(EPHEM, 'logs', 'stage3.log')),
    ],
)

from wita_v2.configs.default import (
    Config, DataConfig, EncoderConfig, TrainConfig,
)

# Stage 3 locked hyperparams — DO NOT MODIFY per-fold.
T_NATIVE         = 32
IMAGE_SIZE       = 336
TEMPORAL_CONTEXT = True
VISIBILITY_GATE  = True
MULTI_JOINT      = False         # set True for the ablation
BELL_SIGMA       = 1.0

UPSAMPLE         = 2
D_MODEL          = 256
N_LAYERS         = 4
N_HEADS          = 4
CONV_KERNEL      = 15
DROPOUT          = 0.2
BATCH_SIZE       = 32
LR_PEAK          = 5e-4
WEIGHT_DECAY     = 5e-2
GRAD_CLIP        = 1.0
NUM_EPOCHS       = 80
WARMUP_PCT       = 0.05
SEED             = 42
FOLDS            = list(range(5))
VARIANT_NAME     = 'stage3_multijoint' if MULTI_JOINT else 'stage3'

# EncoderConfig.arch='siglip' is a benign placeholder — the trainer reads
# features straight from the Stage 3 cache and never touches cfg.encoder.
cfg = Config(
    data=DataConfig(
        hf_repo_id='yewon816/WiTA', lang='english',
        max_zips=None, max_frames=64,
        train_split=0.90, seed=SEED,
        download_dir=os.path.join(EPHEM, 'hf_downloads'),
    ),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
        num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED,
        checkpoint_dir=os.path.join(EPHEM, 'checkpoints'),
    ),
).build()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

print(f'Device         : {cfg.device}')
print(f'Stage 3 cache  : {STAGE3_CACHE}')
print(f'CV manifest    : {CV_MANIFEST}')
print(f'Results path   : {RESULTS_PATH}')
print(f'Variant        : {VARIANT_NAME}')
print(f'Multi-joint    : {MULTI_JOINT}')
print(f'Folds          : {FOLDS}')
print(f'Image size     : {IMAGE_SIZE}x{IMAGE_SIZE}  (24x24 patch grid for dinov2-small)')

## Cell 3 — Stream WiTA data (only if Stage 3 cache doesn't yet exist)

In [ ]:
from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
if os.path.exists(STAGE3_CACHE):
    print(f'Cache exists at {STAGE3_CACHE} — skipping stream/extraction.')
    samples = None
else:
    samples = stream_and_index_with_subjects(cfg)
    print(f'Total: {len(samples)} clips across {len({s for _,_,s in samples})} subjects')

## Cell 4 — Build (or load) the Stage 3 fingertip-pool cache

In [ ]:
from wita_v2.models.encoders.dinov2_encoder import DINOv2Encoder
from wita_v2.datasets.dinov2_feature_cache import (
    extract_dinov2_feature_cache, make_fingerprint, check_fingerprint_match,
)

if not os.path.exists(STAGE3_CACHE):
    encoder = DINOv2Encoder(
        model_name='facebook/dinov2-small',
        image_size=IMAGE_SIZE, pool='mean_patch',  # pool flag unused in patch path
    )
    extract_dinov2_feature_cache(
        samples=samples, encoder=encoder, out_path=STAGE3_CACHE,
        T_native=T_NATIVE,
        temporal_context=TEMPORAL_CONTEXT,
        visibility_gate=VISIBILITY_GATE,
        multi_joint=MULTI_JOINT,
        bell_sigma=BELL_SIGMA,
        device=cfg.device, dtype=torch.float16,
    )
    del encoder
    torch.cuda.empty_cache()

cache = torch.load(STAGE3_CACHE, map_location='cpu', weights_only=False)
expected_fp = {
    'model_name': 'facebook/dinov2-small',
    'image_size': IMAGE_SIZE,
    'grid_size':  IMAGE_SIZE // 14,
    'patch_size': 14,
    'pool':       'fingertip_3x3_bell',
    'temporal_context': TEMPORAL_CONTEXT,
    'visibility_gate':  VISIBILITY_GATE,
    'multi_joint':      MULTI_JOINT,
}
check_fingerprint_match(cache, expected_fp)
print(f'Loaded cache: {len(cache["feats"])} clips, '
      f'in_dim={cache["out_dim"]}, '
      f'detect_rate={cache.get("frame_detect_rate", 0)*100:.1f}%')

## Cell 5 — Build (or load) 5-fold subject CV manifest

We re-use the SAME `subject_cv5.json` manifest as Stage 1 v3 so fold splits are pairwise comparable across stages.

In [ ]:
from wita_v2.datasets.cv_splits import build_cv5_manifest, save_cv5_manifest, load_cv5_manifest

if os.path.exists(CV_MANIFEST):
    manifest = load_cv5_manifest(CV_MANIFEST)
    print(f'Loaded CV manifest from {CV_MANIFEST}')
else:
    fake_samples = [(b'', cache['labels'][i], cache['subjects'][i])
                    for i in range(len(cache['feats']))]
    manifest = build_cv5_manifest(fake_samples, n_folds=5, seed=SEED)
    save_cv5_manifest(manifest, CV_MANIFEST)

print(f'\nCV manifest: {manifest["n_subjects_total"]} signers, {manifest["n_folds"]} folds')
for f in manifest['folds']:
    print(f'  fold {f["fold"]}: train={f["n_train_subjects"]} signers ({f["n_train_clips"]} clips)  '
          f'val={f["n_val_subjects"]} signers ({f["n_val_clips"]} clips)')

## Cell 6 — Run the 5-fold sweep

Each fold uses the same `train_one_fold` function from `training/stage3_train.py`. Results are written to disk incrementally so the notebook tolerates session disconnects.

Loop order: one variant, fold-major (5 runs total).  Resume logic skips completed folds.

In [ ]:
from wita_v2.training.stage3_train import train_one_fold
from wita_v2.datasets.cv_splits     import fold_indices
from wita_v2.datasets.skeleton_augment import LandmarkAugment

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} folds already complete.')
else:
    all_results = []
    completed = set()

# Temporal-only augmentation — spatial jitter disabled per Stage 3 contract.
train_aug = LandmarkAugment(p_spatial_jitter=0.0, spatial_sigma=0.0)

subject_per_idx = cache['subjects']
for fold in FOLDS:
    if (fold, VARIANT_NAME) in completed:
        continue
    train_idx, val_idx = fold_indices(manifest, fold, subject_per_idx)
    result = train_one_fold(
        cache=cache, train_idx=train_idx, val_idx=val_idx, cfg=cfg,
        fold=fold, variant=VARIANT_NAME,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE,
        warmup_pct=WARMUP_PCT, transform=train_aug,
    )
    summary = {k: v for k, v in result.items() if k != 'history'}
    all_results.append(summary)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  saved → {RESULTS_PATH}  (total folds done: {len(all_results)})\n')

print(f'\nAll {len(all_results)}/{len(FOLDS)} folds done.')

## Cell 7 — Aggregate: mean ± std across folds

In [ ]:
import numpy as np

with open(RESULTS_PATH) as f:
    all_results = json.load(f)

per_fold = {r['fold']: r for r in all_results if r['variant'] == VARIANT_NAME}
best_cers = [per_fold[f]['best_val_cer'] for f in FOLDS if f in per_fold]
final_nlls = [per_fold[f]['final_train_nll'] for f in FOLDS if f in per_fold]
nll_below = [per_fold[f]['train_nll_ever_below_05'] for f in FOLDS if f in per_fold]

print(f'Stage 3 ({VARIANT_NAME}) — {len(best_cers)} folds complete\n')
print(' fold    best_val_cer   final_train_nll    train_nll<0.5')
print(' ----    ------------   ---------------    -------------')
for f in FOLDS:
    if f not in per_fold:
        print(f'  {f:>2d}    {"-":>10}     {"-":>13}      {"-":>10}')
        continue
    r = per_fold[f]
    print(f'  {f:>2d}    {r["best_val_cer"]:.4f}        {r["final_train_nll"]:.4f}            '
          f'{"yes" if r["train_nll_ever_below_05"] else "NO"}')
if best_cers:
    arr = np.array(best_cers)
    print(f'\nMean ± std val CER : {arr.mean():.4f} ± {arr.std(ddof=0):.4f}  (n={len(arr)})')

## Cell 8 — Verdict per Stage 3 pass/fail thresholds

In [ ]:
if best_cers:
    arr = np.array(best_cers)
    mean_cer = arr.mean()
    print('\n=== Stage 3 verdict ===')
    print(f'  variant            : {VARIANT_NAME}')
    print(f'  mean ± std val CER : {mean_cer:.4f} ± {arr.std(ddof=0):.4f}')
    if not all(nll_below):
        n_fail = sum(1 for b in nll_below if not b)
        print(f'  ❌ GATING TEST FAILED: {n_fail}/{len(nll_below)} folds did NOT reach train NLL < 0.5.')
        print('     Per plan §2 Task B, the fingertip-pool design is feature-insufficient.')
        print('     Investigate before running Stage 4 — fusion will not save it.')
    elif mean_cer <= 0.78:
        print(f'  ✅ STRETCH ({mean_cer:.4f} ≤ 0.78) — Stage 4 fusion is likely to clear 0.55.')
    elif mean_cer <= 0.860:
        print(f'  ✅ HEADLINE ({mean_cer:.4f} ≤ 0.860) — beats Stage 2 mean.  Proceed to Stage 4.')
    else:
        print(f'  ❌ FAIL: mean {mean_cer:.4f} > 0.860 — does not beat Stage 2.')
        print('     Inspect train NLL gap, blank prob, and per-signer scatter before proceeding.')

## Cell 9 — Per-signer val CER scatter (with easy/hard regime lines)

Shows whether Stage 3 closes the hard-regime tail (PHW, PJH, SYB, KJM, KNY, LKS, KIM, YMG) or whether the easy regime carries the gain.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

per_signer = {}
for r in all_results:
    if r['variant'] != VARIANT_NAME: continue
    per_signer.update(r.get('best_per_signer_val_cer', {}) or {})
items = sorted(per_signer.items(), key=lambda kv: kv[1])
xs = list(range(len(items)))
ys = [v for _, v in items]
HARD = ['PHW','PJH','SYB','KJM','KNY','LKS','KIM','YMG']
colors = ['#d62728' if s in HARD else '#1f77b4' for s,_ in items]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(xs, ys, s=30, c=colors)
ax.axhline(0.55, color='green', linestyle='--', alpha=0.5, label='easy ≤ 0.55')
ax.axhline(0.75, color='red',   linestyle='--', alpha=0.5, label='hard ≥ 0.75')
ax.set_xticks(xs)
ax.set_xticklabels([s for s,_ in items], rotation=90, fontsize=7)
ax.set_ylabel('val CER at best-epoch')
ax.set_xlabel('signer (sorted by CER)')
ax.set_title(f'Stage 3 ({VARIANT_NAME}) — per-signer val CER across 5 folds')
ax.legend(frameon=False)
ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EPHEM, 'logs', 'stage3_per_signer_scatter.png'), dpi=140)
plt.show()